In [65]:
# ============================================================
# 0) IMPORTLAR
# ============================================================

import os
import re
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor

In [67]:
# ============================================================
# 1) DOSYA YOLLARI
# ============================================================

MODEL_GIRIS_PATH = "model_giris.csv"
AGE_PATH = "2025nufus-yas.csv"
HEKIM_PATH = "1000kisibasinahekim.csv"
YATAK_PATH = "100binkisibasinahastaneyatak.csv"

FORECAST_YEARS = list(range(2025, 2031))

In [68]:
# 2) YARDIMCI FONKSİYONLAR
# ============================================================

TR_MAP = str.maketrans({
    "İ": "i", "I": "i", "ı": "i",
    "Ş": "s", "ş": "s",
    "Ğ": "g", "ğ": "g",
    "Ü": "u", "ü": "u",
    "Ö": "o", "ö": "o",
    "Ç": "c", "ç": "c",
})

def normalize_city_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"-\d+$", "", x)
    x = x.translate(TR_MAP).lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def clean_numeric(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace(",", ".")
    x = re.sub(r"[^0-9.\-]", "", x)
    if x in ["", "-", ".", "-."]:
        return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan

def read_pipe_lines(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return [line.strip() for line in f if line.strip()]

In [69]:
# ============================================================
# 3) MODEL_GIRIS OKU
# ============================================================

df_main = pd.read_csv(MODEL_GIRIS_PATH)
df_main.columns = [c.strip() for c in df_main.columns]

df_main["yıl"] = pd.to_numeric(df_main["yıl"], errors="coerce").astype(int)
df_main["il_key"] = df_main["il_adı"].apply(normalize_city_name)
df_main["nufus"] = pd.to_numeric(df_main["nufus"], errors="coerce")

print("model_giris:", df_main.shape)
display(df_main.head())

model_giris: (1863, 12)


,il_adı,yıl,hastane,yatak,nufus,asistan,diger,ebe,hemsire,pratisyen,uzman,il_key
0,Adana,2002,18.0,4266.0,1916637.0,646.0,1182.0,1110.0,1927.0,826.0,1227.0,adana
1,Adana,2003,21.0,4382.0,1933428.0,794.0,1414.0,1147.0,2118.0,844.0,1439.0,adana
2,Adana,2004,21.0,4207.0,1951142.0,856.0,1469.0,1167.0,2157.0,892.0,1497.0,adana
3,Adana,2005,22.0,4773.0,1969512.0,869.0,1540.0,1186.0,2223.0,979.0,1555.0,adana
4,Adana,2006,21.0,4973.0,1988277.0,409.0,1683.0,1258.0,2235.0,938.0,1364.0,adana


In [72]:
# ============================================================
# 4) YAŞ DOSYASINI PARSE ET
# 2007-2025 yaş grubu sayıları
# ============================================================

def parse_age_file(path):
    rows = []
    current_year = None

    lines = read_pipe_lines(path)

    for line in lines:
        parts = line.split("|")

        first = parts[0].strip() if len(parts) > 0 else ""
        second = parts[1].strip() if len(parts) > 1 else ""

        if first.isdigit():
            current_year = int(first)
            city_raw = second
            values = parts[2:]
        elif current_year is not None and second:
            city_raw = second
            values = parts[2:]
        else:
            continue

        if len(values) < 9:
            continue

        rows.append({
            "yıl": current_year,
            "il_adı_age": re.sub(r"-\d+$", "", city_raw).strip(),
            "il_key": normalize_city_name(city_raw),
            "yas_0_4": clean_numeric(values[0]),
            "yas_10_14": clean_numeric(values[1]),
            "yas_5_9": clean_numeric(values[2]),
            "yas_65_69": clean_numeric(values[3]),
            "yas_70_74": clean_numeric(values[4]),
            "yas_75_79": clean_numeric(values[5]),
            "yas_80_84": clean_numeric(values[6]),
            "yas_85_89": clean_numeric(values[7]),
            "yas_90_plus": clean_numeric(values[8]),
        })

    age = pd.DataFrame(rows)

    if age.empty:
        raise ValueError("Yaş dosyası parse edilemedi.")

    age["yıl"] = age["yıl"].astype(int)

    return age

df_age = parse_age_file(AGE_PATH)

print("yaş dosyası:", df_age.shape)
print("Yaş yıl aralığı:", df_age["yıl"].min(), "-", df_age["yıl"].max())
print("Yaş il sayısı:", df_age["il_key"].nunique())
display(df_age.head())


yaş dosyası: (1539, 12)
Yaş yıl aralığı: 2007 - 2025
Yaş il sayısı: 81


,yıl,il_adı_age,il_key,yas_0_4,yas_10_14,yas_5_9,yas_65_69,yas_70_74,yas_75_79,yas_80_84,yas_85_89,yas_90_plus
0,2007,Adana,adana,170903.0,196529.0,192331.0,40794.0,30617.0,23068.0,12182.0,4004.0,2430.0
1,2007,Adıyaman,adiyaman,60457.0,69026.0,65992.0,11596.0,8838.0,7487.0,3828.0,1449.0,838.0
2,2007,Afyonkarahisar,afyonkarahisar,55658.0,61309.0,61660.0,21446.0,18284.0,14187.0,6239.0,1547.0,691.0
3,2007,Ağrı,agri,66964.0,70488.0,82191.0,6722.0,5069.0,3804.0,2067.0,827.0,318.0
4,2007,Aksaray,aksaray,32596.0,36850.0,37386.0,8575.0,6885.0,4498.0,2465.0,841.0,441.0


In [73]:
# ============================================================
# 5) GENİŞ FORMATTAKİ HEKİM/YATAK ORAN DOSYALARINI PARSE ET
# ============================================================

def parse_wide_indicator_file(path, value_name):
    lines = read_pipe_lines(path)

    city_parts = None
    for line in lines:
        parts = line.split("|")
        if "Satırlar" in parts[0]:
            city_parts = parts[3:]
            break

    if city_parts is None:
        raise ValueError(f"Şehir başlığı bulunamadı: {path}")

    cities = []
    for c in city_parts:
        c = c.strip()
        if c:
            cities.append(c)

    rows = []

    for line in lines:
        parts = line.split("|")

        year = None
        value_start_idx = None

        for idx, p in enumerate(parts):
            p_clean = p.strip()
            if p_clean.isdigit() and len(p_clean) == 4:
                year = int(p_clean)
                value_start_idx = idx + 1
                break

        if year is None:
            continue

        values = parts[value_start_idx:value_start_idx + len(cities)]

        for city_raw, value in zip(cities, values):
            rows.append({
                "yıl": year,
                "il_adı": re.sub(r"-\d+$", "", city_raw).strip(),
                "il_key": normalize_city_name(city_raw),
                value_name: clean_numeric(value)
            })

    out = pd.DataFrame(rows)

    if out.empty:
        raise ValueError(f"Veri parse edilemedi: {path}")

    out["yıl"] = out["yıl"].astype(int)

    return out

df_hekim = parse_wide_indicator_file(HEKIM_PATH, "hekim_oran")
df_yatak = parse_wide_indicator_file(YATAK_PATH, "yatak_oran")

print("hekim oran:", df_hekim.shape, df_hekim["yıl"].min(), "-", df_hekim["yıl"].max())
print("yatak oran:", df_yatak.shape, df_yatak["yıl"].min(), "-", df_yatak["yıl"].max())

display(df_hekim.head())
display(df_yatak.head())

hekim oran: (1863, 4) 2002 - 2024
yatak oran: (1863, 4) 2002 - 2024


,yıl,il_adı,il_key,hekim_oran
0,2002,Adana,adana,1.4
1,2002,Adıyaman,adiyaman,0.6
2,2002,Afyonkarahisar,afyonkarahisar,0.8
3,2002,Aksaray,aksaray,1.0
4,2002,Amasya,amasya,1.0


,yıl,il_adı,il_key,yatak_oran
0,2002,Adana,adana,226.3
1,2002,Adıyaman,adiyaman,123.9
2,2002,Afyonkarahisar,afyonkarahisar,229.4
3,2002,Aksaray,aksaray,126.1
4,2002,Amasya,amasya,276.5


In [74]:
# ============================================================
# 6) 2025-2030 NÜFUS TABLOSU — HAZIR PROJEKSİYON DOSYASINDAN OKU
# ============================================================

NUFUS_PROJ_PATH = "Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx"

def load_future_population_from_excel(path, forecast_years):
    """
    Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx dosyasındaki
    'ML Formatı' sayfasını okuyarak 2025-2030 il bazlı nüfusları alır.

    2025: Gerçek
    2026-2030: Güncel Tahmin
    """

    pop = pd.read_excel(
        path,
        sheet_name="ML Formatı",
        header=1
    )

    # Sütun adlarını normalize et
    pop = pop.rename(columns={
        "Yıl": "yıl",
        "İl": "il_adı",
        "Nüfus": "nufus",
        "Veri Türü": "nufus_veri_turu",
        "Kaynak": "nufus_kaynak"
    })

    pop["yıl"] = pd.to_numeric(pop["yıl"], errors="coerce").astype(int)
    pop["nufus"] = pd.to_numeric(pop["nufus"], errors="coerce")
    pop["il_key"] = pop["il_adı"].apply(normalize_city_name)

    # Sadece 2025-2030 yıllarını al
    pop = pop[pop["yıl"].isin(forecast_years)].copy()

    # Aynı il-yıl tekrarları varsa temizle
    pop = (
        pop
        .sort_values(["il_key", "yıl"])
        .drop_duplicates(subset=["il_key", "yıl"], keep="last")
        .reset_index(drop=True)
    )

    # Kontrol
    expected_rows = 81 * len(forecast_years)
    print("future_pop satır sayısı:", len(pop))
    print("Beklenen yaklaşık satır sayısı:", expected_rows)
    print("Yıl aralığı:", pop["yıl"].min(), "-", pop["yıl"].max())
    print("İl sayısı:", pop["il_key"].nunique())

    display(
        pop.groupby("yıl")
        .agg(
            il_sayisi=("il_key", "nunique"),
            toplam_nufus=("nufus", "sum"),
            veri_turu=("nufus_veri_turu", lambda x: ", ".join(sorted(set(map(str, x)))))
        )
        .reset_index()
    )

    return pop[[
        "il_adı",
        "il_key",
        "yıl",
        "nufus",
        "nufus_veri_turu",
        "nufus_kaynak"
    ]].copy()


future_pop = load_future_population_from_excel(
    NUFUS_PROJ_PATH,
    FORECAST_YEARS
)

print("future_pop hazır:", future_pop.shape)
display(future_pop.head(10))

future_pop satır sayısı: 486
Beklenen yaklaşık satır sayısı: 486
Yıl aralığı: 2025 - 2030
İl sayısı: 81


,yıl,il_sayisi,toplam_nufus,veri_turu
0,2025,81,86092168,Gerçek
1,2026,81,86323615,Güncel Tahmin
2,2027,81,86616122,Güncel Tahmin
3,2028,81,86896078,Güncel Tahmin
4,2029,81,87163082,Güncel Tahmin
5,2030,81,87416670,Güncel Tahmin


future_pop hazır: (486, 6)


,il_adı,il_key,yıl,nufus,nufus_veri_turu,nufus_kaynak
0,Adana,adana,2025,2283609,Gerçek,TÜİK ADNKS 2025
1,Adana,adana,2026,2293499,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction
2,Adana,adana,2027,2299727,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction
3,Adana,adana,2028,2305551,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction
4,Adana,adana,2029,2310973,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction
5,Adana,adana,2030,2315809,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction
6,Adıyaman,adiyaman,2025,617821,Gerçek,TÜİK ADNKS 2025
7,Adıyaman,adiyaman,2026,623112,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction
8,Adıyaman,adiyaman,2027,629218,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction
9,Adıyaman,adiyaman,2028,634858,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction


In [76]:
# ============================================================
# 7) YAŞ HEDEFLERİNİ OLUŞTUR
# 2025 için nüfus yoksa future_pop içindeki 2025 nüfusunu kullanır.
# ============================================================

age_base = df_age.copy()

age_base = age_base.merge(
    df_main[["il_key", "yıl", "nufus"]],
    on=["il_key", "yıl"],
    how="left"
)

age_base = age_base.merge(
    future_pop[["il_key", "yıl", "nufus"]],
    on=["il_key", "yıl"],
    how="left",
    suffixes=("", "_future")
)

age_base["nufus"] = age_base["nufus"].combine_first(age_base["nufus_future"])
age_base = age_base.drop(columns=["nufus_future"])

age_base["yas_0_14"] = (
    age_base["yas_0_4"] +
    age_base["yas_5_9"] +
    age_base["yas_10_14"]
) / age_base["nufus"] * 100

age_base["yas_65_plus"] = (
    age_base["yas_65_69"] +
    age_base["yas_70_74"] +
    age_base["yas_75_79"] +
    age_base["yas_80_84"] +
    age_base["yas_85_89"] +
    age_base["yas_90_plus"]
) / age_base["nufus"] * 100

# il_adı için model_giris'teki isimleri kullan
city_names = df_main[["il_key", "il_adı"]].drop_duplicates()
age_base = age_base.merge(city_names, on="il_key", how="left")
age_base["il_adı"] = age_base["il_adı"].combine_first(age_base["il_adı_age"])

age_base = age_base[
    ["il_adı", "il_key", "yıl", "nufus", "yas_0_14", "yas_65_plus"]
].copy()

print("age_base:", age_base.shape)
print("age_base yıl aralığı:", age_base["yıl"].min(), "-", age_base["yıl"].max())
display(age_base.head())




age_base: (1539, 6)
age_base yıl aralığı: 2007 - 2025


,il_adı,il_key,yıl,nufus,yas_0_14,yas_65_plus
0,Adana,adana,2007,2006650.0,27.895398,5.636010
1,Adıyaman,adiyaman,2007,582762.0,33.542853,5.840463
2,Afyonkarahisar,afyonkarahisar,2007,701572.0,25.460965,8.893456
3,Ağrı,agri,2007,530879.0,41.373458,3.542615
4,Aksaray,aksaray,2007,366109.0,29.180381,6.474848


In [77]:
# ============================================================
# 8) ORAN HEDEFLERİNİ OLUŞTUR
# hekim_oran ve yatak_oran zaten dosyalarda doğrudan var.
# ============================================================

ratio_base = df_hekim.merge(
    df_yatak,
    on=["il_adı", "il_key", "yıl"],
    how="outer"
)

ratio_base = ratio_base.merge(
    df_main[["il_key", "yıl", "nufus"]],
    on=["il_key", "yıl"],
    how="left"
)

ratio_base = ratio_base[
    ["il_adı", "il_key", "yıl", "nufus", "hekim_oran", "yatak_oran"]
].copy()

print("ratio_base:", ratio_base.shape)
print("ratio_base yıl aralığı:", ratio_base["yıl"].min(), "-", ratio_base["yıl"].max())
display(ratio_base.head())




ratio_base: (1863, 6)
ratio_base yıl aralığı: 2002 - 2024


,il_adı,il_key,yıl,nufus,hekim_oran,yatak_oran
0,Adana,adana,2002,1916637.0,1.4,226.3
1,Adana,adana,2003,1933428.0,1.6,230.3
2,Adana,adana,2004,1951142.0,1.7,219.1
3,Adana,adana,2005,1969512.0,1.7,245.9
4,Adana,adana,2006,1988277.0,1.4,253.6


In [78]:
# ============================================================
# 9) MODEL VE FEATURE FONKSİYONLARI - DÜZELTİLMİŞ BUILD_PIPELINE
# ============================================================

def add_time_features(data):
    d = data.copy()
    d["yıl"] = d["yıl"].astype(int)
    d = d.sort_values(["il_key", "yıl"]).reset_index(drop=True)
    d["log_nufus"] = np.log1p(d["nufus"])
    min_year_by_city = d.groupby("il_key")["yıl"].transform("min")
    d["city_time_index"] = d["yıl"] - min_year_by_city + 1
    return d


def add_target_lag_features(data, target):
    d = data.copy()
    d = d.sort_values(["il_key", "yıl"]).reset_index(drop=True)

    g = d.groupby("il_key")[target]

    d[f"{target}_lag1"] = g.shift(1)
    d[f"{target}_lag2"] = g.shift(2)
    d[f"{target}_lag3"] = g.shift(3)

    d[f"{target}_roll3"] = (
        g.shift(1)
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    d[f"{target}_diff1"] = d[f"{target}_lag1"] - d[f"{target}_lag2"]

    return d


def get_feature_cols(target):
    categorical_features = ["il_adı"]

    numeric_features = [
        "yıl",
        "nufus",
        "log_nufus",
        "city_time_index",
        f"{target}_lag1",
        f"{target}_lag2",
        f"{target}_lag3",
        f"{target}_roll3",
        f"{target}_diff1",
    ]

    return categorical_features, numeric_features


def make_onehot_encoder():
    """
    Farklı sklearn sürümlerine uyumlu OneHotEncoder.
    Yeni sürümlerde sparse_output=False,
    eski sürümlerde sparse=False kullanılır.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_pipeline(model, target):
    categorical_features, numeric_features = get_feature_cols(target)

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                make_onehot_encoder(),
                categorical_features
            ),
            (
                "num",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_features
            )
        ],
        remainder="drop",
        sparse_threshold=0
    )

    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", clone(model))
    ])

    feature_cols = categorical_features + numeric_features

    return pipe, feature_cols

In [79]:
# ============================================================
# 10) RECURSIVE FORECAST FONKSİYONU
# ============================================================

def recursive_forecast_target(
    base_data,
    future_pop,
    target,
    model,
    forecast_years,
    keep_actual_years=None
):
    if keep_actual_years is None:
        keep_actual_years = []

    required_cols = ["il_adı", "il_key", "yıl", "nufus", target]

    missing_cols = [c for c in required_cols if c not in base_data.columns]
    if missing_cols:
        raise ValueError(f"{target} için eksik sütunlar: {missing_cols}")

    hist = base_data[required_cols].copy()
    hist["yıl"] = hist["yıl"].astype(int)

    future = future_pop[["il_adı", "il_key", "yıl", "nufus"]].copy()
    future[target] = np.nan

    actual_future_values = hist[
        hist["yıl"].isin(forecast_years) &
        hist[target].notna()
    ][["il_key", "yıl", target]].copy()

    if len(actual_future_values) > 0:
        future = future.merge(
            actual_future_values,
            on=["il_key", "yıl"],
            how="left",
            suffixes=("", "_actual")
        )
        future[target] = future[f"{target}_actual"].combine_first(future[target])
        future = future.drop(columns=[f"{target}_actual"])

    train_base = hist[hist[target].notna()].copy()
    train_base = add_time_features(train_base)
    train_base = add_target_lag_features(train_base, target)

    categorical_features, numeric_features = get_feature_cols(target)
    feature_cols = categorical_features + numeric_features

    train_data = train_base.dropna(subset=[target, f"{target}_lag1"]).copy()

    X_train = train_data[feature_cols]
    y_train = train_data[target]

    pipe, feature_cols = build_pipeline(model, target)
    pipe.fit(X_train, y_train)

    combined = pd.concat(
        [
            hist[["il_adı", "il_key", "yıl", "nufus", target]],
            future[["il_adı", "il_key", "yıl", "nufus", target]]
        ],
        ignore_index=True
    )

    combined = (
        combined
        .sort_values(["il_key", "yıl"])
        .drop_duplicates(subset=["il_key", "yıl"], keep="last")
        .reset_index(drop=True)
    )

    prediction_rows = []

    for year in forecast_years:
        combined = add_time_features(combined)
        combined = add_target_lag_features(combined, target)

        mask_year = combined["yıl"] == year

        if year in keep_actual_years:
            actual_rows = combined[mask_year].copy()
            actual_rows["prediction_type"] = "actual"
            prediction_rows.append(
                actual_rows[["il_adı", "il_key", "yıl", "nufus", target, "prediction_type"]]
            )
            continue

        mask_pred = mask_year & combined[target].isna()

        if mask_pred.sum() > 0:
            X_future = combined.loc[mask_pred, feature_cols].copy()
            y_pred = pipe.predict(X_future)
            y_pred = np.maximum(y_pred, 0)
            combined.loc[mask_pred, target] = y_pred

        pred_rows = combined[mask_year].copy()
        pred_rows["prediction_type"] = np.where(
            pred_rows[target].notna(),
            "predicted",
            "missing"
        )

        prediction_rows.append(
            pred_rows[["il_adı", "il_key", "yıl", "nufus", target, "prediction_type"]]
        )

    result = pd.concat(prediction_rows, ignore_index=True)
    result = result.sort_values(["il_key", "yıl"]).reset_index(drop=True)

    return result, pipe



In [80]:

# ============================================================
# 11) HER HEDEF İÇİN 2025-2030 TAHMİN
# ============================================================

forecast_outputs = {}
trained_final_models = {}

target_base_map = {
    "yas_0_14": age_base,
    "yas_65_plus": age_base,
    "hekim_oran": ratio_base,
    "yatak_oran": ratio_base,
}

for target, model in BEST_MODEL_MAP.items():
    print()
    print("=" * 100)
    print(f"HEDEF TAHMİNİ: {target}")
    print("=" * 100)

    if target in ["yas_0_14", "yas_65_plus"]:
        keep_years = [2025]
    else:
        keep_years = []

    target_forecast, trained_pipe = recursive_forecast_target(
        base_data=target_base_map[target],
        future_pop=future_pop,
        target=target,
        model=model,
        forecast_years=FORECAST_YEARS,
        keep_actual_years=keep_years
    )

    forecast_outputs[target] = target_forecast
    trained_final_models[target] = trained_pipe

    print(target_forecast.shape)
    display(target_forecast.head(12))






HEDEF TAHMİNİ: yas_0_14
(486, 6)


,il_adı,il_key,yıl,nufus,yas_0_14,prediction_type
0,Adana,adana,2025,2283609.0,21.993213,actual
1,Adana,adana,2026,2293499.0,21.286629,predicted
2,Adana,adana,2027,2299727.0,20.825061,predicted
3,Adana,adana,2028,2305551.0,20.374964,predicted
4,Adana,adana,2029,2310973.0,19.750900,predicted
5,Adana,adana,2030,2315809.0,18.924919,predicted
6,Adıyaman,adiyaman,2025,617821.0,26.074866,actual
7,Adıyaman,adiyaman,2026,623112.0,25.344479,predicted
8,Adıyaman,adiyaman,2027,629218.0,24.622162,predicted
9,Adıyaman,adiyaman,2028,634858.0,24.025957,predicted



HEDEF TAHMİNİ: yas_65_plus
(486, 6)


,il_adı,il_key,yıl,nufus,yas_65_plus,prediction_type
0,Adana,adana,2025,2283609.0,10.586751,actual
1,Adana,adana,2026,2293499.0,11.016234,predicted
2,Adana,adana,2027,2299727.0,11.457670,predicted
3,Adana,adana,2028,2305551.0,11.919366,predicted
4,Adana,adana,2029,2310973.0,12.396739,predicted
5,Adana,adana,2030,2315809.0,12.889246,predicted
6,Adıyaman,adiyaman,2025,617821.0,9.208169,actual
7,Adıyaman,adiyaman,2026,623112.0,9.578499,predicted
8,Adıyaman,adiyaman,2027,629218.0,9.968357,predicted
9,Adıyaman,adiyaman,2028,634858.0,10.371187,predicted



HEDEF TAHMİNİ: hekim_oran
(486, 6)


,il_adı,il_key,yıl,nufus,hekim_oran,prediction_type
0,Adana,adana,2025,2283609.0,2.602272,predicted
1,Adana,adana,2026,2293499.0,2.631855,predicted
2,Adana,adana,2027,2299727.0,2.657828,predicted
3,Adana,adana,2028,2305551.0,2.694466,predicted
4,Adana,adana,2029,2310973.0,2.736625,predicted
5,Adana,adana,2030,2315809.0,2.783504,predicted
6,Adıyaman,adiyaman,2025,617821.0,1.694355,predicted
7,Adıyaman,adiyaman,2026,623112.0,1.758969,predicted
8,Adıyaman,adiyaman,2027,629218.0,1.830562,predicted
9,Adıyaman,adiyaman,2028,634858.0,1.894149,predicted



HEDEF TAHMİNİ: yatak_oran
(486, 6)


,il_adı,il_key,yıl,nufus,yatak_oran,prediction_type
0,Adana,adana,2025,2283609.0,323.985432,predicted
1,Adana,adana,2026,2293499.0,329.220907,predicted
2,Adana,adana,2027,2299727.0,334.550691,predicted
3,Adana,adana,2028,2305551.0,339.729930,predicted
4,Adana,adana,2029,2310973.0,344.715297,predicted
5,Adana,adana,2030,2315809.0,349.556599,predicted
6,Adıyaman,adiyaman,2025,617821.0,188.844754,predicted
7,Adıyaman,adiyaman,2026,623112.0,195.163020,predicted
8,Adıyaman,adiyaman,2027,629218.0,201.209825,predicted
9,Adıyaman,adiyaman,2028,634858.0,207.213449,predicted


In [81]:
# ============================================================
# 12) YAŞ TABLOSU
# 2025 gerçek/sabit + 2026-2030 tahmin
# ============================================================

yas_0_14_table = forecast_outputs["yas_0_14"][
    ["il_adı", "il_key", "yıl", "yas_0_14", "prediction_type"]
].rename(columns={"prediction_type": "yas_0_14_tip"})

yas_65_table = forecast_outputs["yas_65_plus"][
    ["il_adı", "il_key", "yıl", "yas_65_plus", "prediction_type"]
].rename(columns={"prediction_type": "yas_65_plus_tip"})

age_forecast_table = yas_0_14_table.merge(
    yas_65_table,
    on=["il_adı", "il_key", "yıl"],
    how="outer"
)

age_forecast_table = age_forecast_table.sort_values(["il_key", "yıl"]).reset_index(drop=True)

print()
print("========== YAŞ TABLOSU: 2025 GERÇEK + 2026-2030 TAHMİN ==========")
display(age_forecast_table.head(20).round(4))




========== YAŞ TABLOSU: 2025 GERÇEK + 2026-2030 TAHMİN ==========


,il_adı,il_key,yıl,yas_0_14,yas_0_14_tip,yas_65_plus,yas_65_plus_tip
0,Adana,adana,2025,21.9932,actual,10.5868,actual
1,Adana,adana,2026,21.2866,predicted,11.0162,predicted
2,Adana,adana,2027,20.8251,predicted,11.4577,predicted
3,Adana,adana,2028,20.3750,predicted,11.9194,predicted
4,Adana,adana,2029,19.7509,predicted,12.3967,predicted
5,Adana,adana,2030,18.9249,predicted,12.8892,predicted
6,Adıyaman,adiyaman,2025,26.0749,actual,9.2082,actual
7,Adıyaman,adiyaman,2026,25.3445,predicted,9.5785,predicted
8,Adıyaman,adiyaman,2027,24.6222,predicted,9.9684,predicted
9,Adıyaman,adiyaman,2028,24.0260,predicted,10.3712,predicted


In [82]:
# ============================================================
# 13) ORAN TABLOSU
# 2025-2030 tahmin
# ============================================================

hekim_table = forecast_outputs["hekim_oran"][
    ["il_adı", "il_key", "yıl", "hekim_oran", "prediction_type"]
].rename(columns={"prediction_type": "hekim_oran_tip"})

yatak_table = forecast_outputs["yatak_oran"][
    ["il_adı", "il_key", "yıl", "yatak_oran", "prediction_type"]
].rename(columns={"prediction_type": "yatak_oran_tip"})

oran_forecast_table = hekim_table.merge(
    yatak_table,
    on=["il_adı", "il_key", "yıl"],
    how="outer"
)

oran_forecast_table = oran_forecast_table.sort_values(["il_key", "yıl"]).reset_index(drop=True)

print()
print("========== ORAN TABLOSU: 2025-2030 TAHMİN ==========")
display(oran_forecast_table.head(20).round(4))






========== ORAN TABLOSU: 2025-2030 TAHMİN ==========


,il_adı,il_key,yıl,hekim_oran,hekim_oran_tip,yatak_oran,yatak_oran_tip
0,Adana,adana,2025,2.6023,predicted,323.9854,predicted
1,Adana,adana,2026,2.6319,predicted,329.2209,predicted
2,Adana,adana,2027,2.6578,predicted,334.5507,predicted
3,Adana,adana,2028,2.6945,predicted,339.7299,predicted
4,Adana,adana,2029,2.7366,predicted,344.7153,predicted
5,Adana,adana,2030,2.7835,predicted,349.5566,predicted
6,Adıyaman,adiyaman,2025,1.6944,predicted,188.8448,predicted
7,Adıyaman,adiyaman,2026,1.7590,predicted,195.1630,predicted
8,Adıyaman,adiyaman,2027,1.8306,predicted,201.2098,predicted
9,Adıyaman,adiyaman,2028,1.8941,predicted,207.2134,predicted


In [83]:

# ============================================================
# 14) FINAL TEK TABLO
# ============================================================

final_forecast_table = future_pop[
    ["il_adı", "il_key", "yıl", "nufus", "nufus_veri_turu", "nufus_kaynak"]
].copy()

final_forecast_table = final_forecast_table.merge(
    age_forecast_table,
    on=["il_adı", "il_key", "yıl"],
    how="left"
)

final_forecast_table = final_forecast_table.merge(
    oran_forecast_table,
    on=["il_adı", "il_key", "yıl"],
    how="left"
)

final_forecast_table = final_forecast_table[
    [
        "il_adı",
        "il_key",
        "yıl",
        "nufus",
        "nufus_veri_turu",
        "nufus_kaynak",
        "yas_0_14",
        "yas_65_plus",
        "hekim_oran",
        "yatak_oran",
        "yas_0_14_tip",
        "yas_65_plus_tip",
        "hekim_oran_tip",
        "yatak_oran_tip"
    ]
].copy()

final_forecast_table = final_forecast_table.sort_values(["il_key", "yıl"]).reset_index(drop=True)

print()
print("========== FINAL TEK TABLO: 2025-2030 ==========")
display(final_forecast_table.head(30).round(4))




========== FINAL TEK TABLO: 2025-2030 ==========


,il_adı,il_key,yıl,nufus,nufus_veri_turu,nufus_kaynak,yas_0_14,yas_65_plus,hekim_oran,yatak_oran,yas_0_14_tip,yas_65_plus_tip,hekim_oran_tip,yatak_oran_tip
0,Adana,adana,2025,2283609,Gerçek,TÜİK ADNKS 2025,21.9932,10.5868,2.6023,323.9854,actual,actual,predicted,predicted
1,Adana,adana,2026,2293499,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,21.2866,11.0162,2.6319,329.2209,predicted,predicted,predicted,predicted
2,Adana,adana,2027,2299727,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,20.8251,11.4577,2.6578,334.5507,predicted,predicted,predicted,predicted
3,Adana,adana,2028,2305551,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,20.3750,11.9194,2.6945,339.7299,predicted,predicted,predicted,predicted
4,Adana,adana,2029,2310973,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,19.7509,12.3967,2.7366,344.7153,predicted,predicted,predicted,predicted
5,Adana,adana,2030,2315809,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,18.9249,12.8892,2.7835,349.5566,predicted,predicted,predicted,predicted
6,Adıyaman,adiyaman,2025,617821,Gerçek,TÜİK ADNKS 2025,26.0749,9.2082,1.6944,188.8448,actual,actual,predicted,predicted
7,Adıyaman,adiyaman,2026,623112,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,25.3445,9.5785,1.7590,195.1630,predicted,predicted,predicted,predicted
8,Adıyaman,adiyaman,2027,629218,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,24.6222,9.9684,1.8306,201.2098,predicted,predicted,predicted,predicted
9,Adıyaman,adiyaman,2028,634858,Güncel Tahmin,TÜİK Ana Senaryo + Bias-Correction,24.0260,10.3712,1.8941,207.2134,predicted,predicted,predicted,predicted


In [84]:
# ============================================================
# 15) KAYDET
# ============================================================

age_forecast_table.to_csv(
    "2025_2030_yas_tahminleri.csv",
    index=False,
    encoding="utf-8-sig"
)

oran_forecast_table.to_csv(
    "2025_2030_hekim_yatak_oran_tahminleri.csv",
    index=False,
    encoding="utf-8-sig"
)

final_forecast_table.to_csv(
    "2025_2030_tum_hedefler_final_tahmin.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("Dosyalar kaydedildi:")
print("2025_2030_yas_tahminleri.csv")
print("2025_2030_hekim_yatak_oran_tahminleri.csv")
print("2025_2030_tum_hedefler_final_tahmin.csv")




Dosyalar kaydedildi:
2025_2030_yas_tahminleri.csv
2025_2030_hekim_yatak_oran_tahminleri.csv
2025_2030_tum_hedefler_final_tahmin.csv


In [85]:
# ============================================================
# 16) KONTROL ÖZETİ
# ============================================================

print()
print("========== KONTROL ÖZETİ ==========")

for target in ["yas_0_14", "yas_65_plus", "hekim_oran", "yatak_oran"]:
    print()
    print("-" * 80)
    print(target)
    print("-" * 80)

    display(
        final_forecast_table
        .groupby("yıl")[target]
        .agg(["count", "mean", "std", "min", "max"])
        .round(4)
    )


========== KONTROL ÖZETİ ==========

--------------------------------------------------------------------------------
yas_0_14
--------------------------------------------------------------------------------


,count,mean,std,min,max
yıl,,,,,
2025,81,20.0145,5.2715,13.5491,36.7898
2026,81,19.5511,5.1205,14.2013,36.1350
2027,81,19.1102,4.9359,14.2128,35.2326
2028,81,18.6746,4.7746,14.1853,34.6874
2029,81,18.2533,4.5718,14.1508,34.1000
2030,81,17.8735,4.3821,14.1542,33.5383



--------------------------------------------------------------------------------
yas_65_plus
--------------------------------------------------------------------------------


,count,mean,std,min,max
yıl,,,,,
2025,81,12.7878,4.2735,3.8041,21.6553
2026,81,13.2137,4.3455,4.0409,22.1551
2027,81,13.6526,4.4150,4.2990,22.6667
2028,81,14.1120,4.4867,4.5726,23.2048
2029,81,14.5875,4.5579,4.8646,23.7582
2030,81,15.0783,4.6281,5.1750,24.3261



--------------------------------------------------------------------------------
hekim_oran
--------------------------------------------------------------------------------


,count,mean,std,min,max
yıl,,,,,
2025,81,2.1557,0.5985,1.2632,4.5065
2026,81,2.2013,0.5719,1.3335,4.4701
2027,81,2.2448,0.5457,1.3987,4.4387
2028,81,2.2944,0.5286,1.4632,4.4365
2029,81,2.3470,0.5164,1.5266,4.4516
2030,81,2.4025,0.5082,1.5891,4.4807



--------------------------------------------------------------------------------
yatak_oran
--------------------------------------------------------------------------------


,count,mean,std,min,max
yıl,,,,,
2025,81,304.5258,81.4148,140.4338,534.9335
2026,81,308.5432,80.6170,147.6079,538.2384
2027,81,312.5867,79.9838,154.3662,541.8600
2028,81,316.6248,79.4582,160.8237,545.4625
2029,81,320.6307,79.0440,166.9009,549.1087
2030,81,324.6153,78.7142,172.6720,552.7923
